# Taller 6 - Punto 3: K-Nearest Neighbors (KNN)

Aplicación: clasificación de especies de flores Iris.

KNN es un algoritmo de aprendizaje supervisado basado en distancia. Para una nueva muestra busca los `k` ejemplos más cercanos y asigna la clase mayoritaria. Como las distancias son sensibles a la escala, se estandarizan las variables antes de entrenar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

print("Features:", X.columns.tolist())
print("Clases:", iris.target_names)
display(X.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

ks = range(1, 21)
scores = []
for k in ks:
    modelo = KNeighborsClassifier(n_neighbors=k)
    scores.append(cross_val_score(modelo, X_train_s, y_train, cv=5, scoring="accuracy").mean())

mejor_k = list(ks)[int(np.argmax(scores))]
print("Mejor k:", mejor_k)
print("Accuracy CV:", max(scores))

plt.figure(figsize=(8,4))
plt.plot(list(ks), scores, marker="o")
plt.xlabel("k")
plt.ylabel("Accuracy promedio")
plt.title("Selección de k mediante validación cruzada")
plt.grid(True)
plt.show()

In [ ]:
knn = KNeighborsClassifier(n_neighbors=mejor_k)
knn.fit(X_train_s, y_train)
pred = knn.predict(X_test_s)

print("Accuracy prueba:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=iris.target_names))

cm = confusion_matrix(y_test, pred)
ConfusionMatrixDisplay(cm, display_labels=iris.target_names).plot()
plt.title("Matriz de confusión - KNN")
plt.show()

In [ ]:
nueva_flor = pd.DataFrame([{
    "sepal length (cm)": 6.1,
    "sepal width (cm)": 2.8,
    "petal length (cm)": 4.7,
    "petal width (cm)": 1.2
}])

nueva_s = scaler.transform(nueva_flor)
clase = knn.predict(nueva_s)[0]
proba = knn.predict_proba(nueva_s)[0]

print("Clase predicha:", iris.target_names[clase])
print("Probabilidades:", dict(zip(iris.target_names, np.round(proba, 3))))